# Symphony's claim protocol, live

This notebook runs the claim protocol's public library, [OpenSpineConsortium/symphony-protocol](https://github.com/OpenSpineConsortium/symphony-protocol) at its release v0.5.1, in a throwaway folder: two workers race for one issue, the push guard judges six pushes, and the board is rendered from a snapshot. It needs no GitHub sign-in, no Claude account and no key of any kind; a local bare repository stands in for GitHub, because what refuses the second claim is git's own rule that a push may not replace history it does not contain, the rule GitHub applies.

Run the cells in order (Runtime, Run all). The same demo runs from a clone of symphony-template as `bash scripts/demo.sh`; docs/start/capabilities.md explains it.

In [ ]:
# Fetch the library at its release tag: public, standard library only, nothing to install.
import json, os, subprocess, sys, tempfile
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="symphony-demo-"))
LIB = WORK / "symphony-protocol"
ENV = dict(os.environ, GIT_CONFIG_GLOBAL="/dev/null", GIT_CONFIG_NOSYSTEM="1",
           PYTHONPATH=str(LIB))
subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "--quiet", "--depth", "1", "--branch", "v0.5.1",
                "https://github.com/OpenSpineConsortium/symphony-protocol", str(LIB)], check=True)

def git(where, *args, check=True):
    return subprocess.run(["git", "-c", "gc.auto=0", "-C", str(where), *args],
                          env=ENV, capture_output=True, text=True, check=check)

print("symphony-protocol", git(LIB, "describe", "--tags").stdout.strip(), "is in", LIB)

## 1. Two workers claim issue #7 at the same moment

A claim is the first push of the branch `claude/<issue>`, whose first commit adds `docs/agents/claims/<issue>.md` naming the worker. Both workers make that commit; whoever reaches origin first holds the issue.

In [ ]:
ORIGIN = WORK / "origin.git"
git(WORK, "init", "--quiet", "--bare", "--initial-branch=main", str(ORIGIN))
seed = WORK / "seed"
git(WORK, "init", "--quiet", "--initial-branch=main", str(seed))
(seed / "README.md").write_text("# demo\n")
git(seed, "add", "README.md")
git(seed, "-c", "user.name=owner", "-c", "user.email=owner@example.invalid", "commit", "-m", "First commit")
git(seed, "remote", "add", "origin", str(ORIGIN))
git(seed, "push", "origin", "main")

def clone_as(name):
    git(WORK, "clone", "--quiet", str(ORIGIN), name)
    git(WORK / name, "config", "user.name", name)
    git(WORK / name, "config", "user.email", f"{name}@example.invalid")

def write_claim(name, issue):
    clone = WORK / name
    git(clone, "switch", "--quiet", "--no-track", "-c", f"claude/{issue}", "origin/main")
    claims = clone / "docs/agents/claims"
    claims.mkdir(parents=True, exist_ok=True)
    (claims / f"{issue}.md").write_text(
        f"# Claim: #{issue}\n\n- Worker: {name}\n- Branch: claude/{issue}\n\n"
        f"## Plan\n\n{name}'s plan.\n\n## Files to touch\n\n- src/widget.py\n")
    git(clone, "add", "docs/agents/claims")
    git(clone, "commit", "-m", f"Claim #{issue}")

def push_claim(name, issue):
    done = git(WORK / name, "push", "origin", f"claude/{issue}", check=False)
    if done.returncode == 0:
        print(f"{name}: push of claude/{issue} accepted, so #{issue} is {name}'s")
    else:
        refused = [l for l in done.stderr.splitlines() if "rejected" in l]
        print(f"{name}: push of claude/{issue} refused by origin: {refused[0].strip()}")
        print(f"   so {name} picks another issue; nobody overwrote anybody")

for name in ("alice", "bob"):
    clone_as(name)
    write_claim(name, 7)
push_claim("alice", 7)
push_claim("bob", 7)

## 2. The push guard judges each push before it runs

In a Symphony repository Claude Code runs the guard before every shell command a session types, and exit code 2 stops the command. Here it judges pushes from bob's clone, after bob has fetched, which is the moment git itself would let a `--force-with-lease` take alice's claim.

In [ ]:
git(WORK / "bob", "fetch", "--quiet", "origin")

def guard(command, clone="bob"):
    payload = json.dumps({"cwd": str(WORK / clone), "tool_input": {"command": command}})
    run = subprocess.run([sys.executable, "-m", "sdlc_protocol", "guard"], input=payload,
                         env=ENV, cwd=WORK / clone, capture_output=True, text=True)
    verdict = "REFUSED" if run.returncode == 2 else "allowed"
    print(f"{verdict:8} {command}")
    if run.stderr.strip():
        print("         " + run.stderr.strip())

for command in ("git push origin main",
                "git push --force origin claude/7",
                "git push origin +claude/7",
                "git push origin claude/07",
                "git push --force-with-lease origin claude/7",
                "git push origin claude/8"):
    guard(command)

## 3. The board, rendered from a snapshot

The board workflow renders STATE.md on every issue, push and pull request event from GitHub's API. `--dry-run` reads the same facts from a JSON snapshot, here one with a claim in progress, a queued task and a blocker. Edit the snapshot and run the cell again to see the board change.

In [ ]:
snapshot = {
  "sha": "0123abcdef0123abcdef0123abcdef0123abcdef",
  "default_branch": "main",
  "issues": [
    {"number": 8, "title": "Write the release notes", "labels": [{"name": "todo"}], "updated_at": "2026-10-20T09:00:00Z"},
    {"number": 9, "title": "CI is red on every branch", "labels": [{"name": "blocker"}], "updated_at": "2026-10-20T09:30:00Z"}
  ],
  "pulls": [
    {"number": 40, "title": "Make the widget faster", "head": {"ref": "claude/7", "sha": "aaaaaaa1111111"},
     "mergeable": True, "mergeable_state": "clean", "updated_at": "2026-10-20T10:00:00Z", "files": ["src/widget.py"]}
  ],
  "branches": ["main", "claude/7"],
  "claims": {"7": "# Claim: #7\n\n- Worker: alice\n- Branch: claude/7\n\n## Plan\n\nalice's plan.\n\n## Files to touch\n\n- src/widget.py\n"},
  "merged": []
}
(WORK / "board.json").write_text(json.dumps(snapshot))
run = subprocess.run([sys.executable, "-m", "sdlc_protocol", "board", "--dry-run", str(WORK / "board.json")],
                     env=ENV, capture_output=True, text=True)
print(run.stdout or run.stderr)

## What this did not show

The parts that need a real repository and an account: the board workflow writing the branch `board`, the coordinator releasing an idle claim after 24 hours, Claude workers in GitHub Actions, and the control panel. docs/start/capabilities.md in symphony-template links each one.